In this notebook we demonstrate how to import the data and perform some exploratory data analysis in an attempt to identify features which are relevant to our modeling goals

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import sys
from pathlib import Path

In [ ]:
pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, MP_THRESHOLD, HOLDOUT_START
import utils

### [Basketball-Reference Bot Policy](https://www.sports-reference.com/bot-traffic.html)

Basketball reference enforces a 1 day block if you are caught accessing the site more than 20 times a second.  All cells that run loops which collect the data should have `time.sleep(x)` in the loop where x $\geq3$.  In what follows we use `time.sleep(3.3)` for good measure.  Currently the cell should take about 8 minutes to run.

#### IF CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()` YOU NEED TO RESTART THE KERNEL, UNCOMMENT THIS CELL, AND RUN ALL.  DON'T FORGET TO RE-COMMENT OUT THIS CELL IF YOU HAVE ALREADY DOWNLOADED THE DATA AND NO CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()` OR YOU WILL BE SAD WHEN YOU RERUN THE NOTEBOOK!

In [ ]:
# utils.scrape_per_game_tables(START_YEAR,END_YEAR)

when we import the datafram from the csv it is important to add a comment keyword or it will mislabel the columns using the descriptive comment instead of the names.

Example for the 2025-2026 season

In [ ]:
season = '2025-2026'
df = pd.read_csv(str(ROOT)+f'/data/{season}_season/per_game_stats_{season}.csv',encoding='utf-8',comment='#')

In [ ]:
df.sort_values('MVPVS', ascending=False)

Note that players who have been traded during the season show up multiple times.  One for each team as well as an aggregate row where the team is listed as "#TM" and # is the number of teams they have been on. We have made the team wins/losses for these rows NaNs.  Here are the people who were on three teams.

In [ ]:
df.loc[df['Team']=='3TM']

Now we import all of the data into a single pandas dataframe

In [ ]:
df_tot = utils.player_stat_dataframe(START_YEAR,END_YEAR)

In [ ]:
df_tot.columns[-4:]

In [ ]:
df_tot.head()

In [ ]:
df_tot.sample(5)

In [ ]:
df_mvp = df_tot.loc[df_tot['MVPVS'] != 0.0].reset_index(drop=True)
df_mvp.sample(5)

df_new = df_tot.copy()
df_new['RecievedVotes'] = (df_tot['MVPVS'] != 0.0).astype('int64')
df_new['TradedMidSeason'] = df_tot['Team'].str.endswith('TM').groupby([df_tot['Year'], df_tot['Code']]).transform('any').astype('int64') #adding a col to show of a player got traded

In [ ]:
target = 'MVPVS'
drop = ['Year', 'Code', 'Player', 'Team', 'Pos', 'Age','FG%', '3P%', '2P%', 'FT%', 'FGA', '3PA', '2PA', 'FTA', 'PF', 'MVP', 'MVPVS']
features = df_mvp.drop(columns=drop).columns.tolist()

Check plots of all possible features against MVP vote share

In [ ]:
for i in range(0,len(features),5):
    sns.pairplot(
        df_tot,
        x_vars=features[i:i+5],
        y_vars=target,
    )
    plt.show()

Look at situations only containing players who received MVP votes

In [ ]:
for i in range(0,len(features),5):
    sns.pairplot(
        df_mvp,
        x_vars=features[i:i+5],
        y_vars=target,
    )
    plt.show()

Make a giant pairplot of all features to be trimmed down

In [ ]:

custom_palette = {
    0: (0.12, 0.47, 0.71, 0.4),  # Light blue
    1: (1.00, 0.50, 0.05, 1.0)   # Solid orange
}
sns.pairplot(
    df_new,
    x_vars=features[0:-1],
    y_vars=features[0:-1],
    hue='RecievedVotes',
    palette=custom_palette,
    plot_kws={"alpha": 0.2,"s":10}
)
plt.show()

Look at some hand-chosen features

In [ ]:

sns.pairplot(
    df_new,
    x_vars=['W%','PTS','FG','DRB','eFG%'],
    y_vars=['W%','PTS','FG','DRB','eFG%'],
    hue='RecievedVotes',
    plot_kws={"s":10},
    palette=custom_palette
)
plt.show()

Make a heatmap to look at correlations between possible features to see what can be easily eliminated

In [ ]:
#heatmap to see correlations

season_rows=df_new[(df_new['TradedMidSeason']==0) | df_new['Team'].str.endswith('TM')]
rel=season_rows[(season_rows['MP']>=MP_THRESHOLD) & (season_rows['G']>=20)]

corr=rel[list(features) + ['MVPVS']].corr(method='spearman')
mask=np.triu(np.ones_like(corr,dtype=bool))
plt.figure(figsize=(12,10))
sns.heatmap(corr,mask=mask,cmap='vlag',center=0,vmin=-1,vmax=1,annot=True,fmt='.2f',annot_kws={'size':7})
plt.title(f'spearman corr--1 row per player-season for MP>={MP_THRESHOLD}')
plt.show()

In [ ]:
#clustering groups features whose rows in heatmap look alike, meaning they correlate with evything else in a similar way. So for feature selection we move up the tree from the leaves and when we hit a group we keep on representative
sns.clustermap(corr,cmap='vlag',center=0,vmin=-1,vmax=1,figsize=(12,12))
plt.show()

the heatmap and clustermap give some sense of stars vs bench players and a big play time effect.

In [ ]:
#addressing my question of finding out among players good enough to get votes, what separates the winner from the rest
vg=rel[rel['MVPVS']>0]
corr_vg=vg[list(features)+['MVPVS']].corr(method='spearman')
mask=np.triu(np.ones_like(corr_vg,dtype=bool))
plt.figure(figsize=(12,10))
sns.heatmap(corr_vg,mask=mask,cmap='vlag',center=0,vmin=-1,vmax=1,annot=True,fmt='.2f',annot_kws={'size':7})
plt.title(f'spearman corr vote getters only (n={len(vg)})')
plt.show()

this heatmap give a sense of what separates MVP players from each other.  Considering it we focus in on the following features to begin with:
```
feats=['PTS','W%','FT','eFG%','DRB','AST','BLK','STL','GS','3P']
```